In [ ]:
import os, sys, django, asyncio, aiohttp

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'beehiiv_analytics.settings')
django.setup()

from django.contrib.auth import get_user_model
from analytics.models import Publication, Post, Section
from analytics.utils import html_to_text_with_links, format_link_history, fetch_post_html

# --- Inputs ---
USER_NAME = "<your-django-username>"       # Django username
PUBLICATION_NAME = "<your-publication-name>"      # Publication.name
POST_TITLE = "<post-title-substring>"             # Post.title (substring match)

# --- Lookups ---
User = get_user_model()
user = User.objects.get(username=USER_NAME)
publication = Publication.objects.get(name=PUBLICATION_NAME)
post = Post.objects.get(
    title__icontains=POST_TITLE,
    user=user,
    publication=publication,
)
usage = user.usageaccount
beehiiv_token = usage.beehiiv_token
beehiiv_pub_id = usage.beehiiv_pub_id

# --- 1) Fetch post HTML from Beehiiv API and extract text ---
async def get_html():
    sem = asyncio.Semaphore(5)
    async with aiohttp.ClientSession() as session:
        _, html = await fetch_post_html(session, post.post_id, sem, beehiiv_token, beehiiv_pub_id)
    return html

html = await get_html()
if not html:
    raise RuntimeError(f"Failed to fetch HTML for post {post.post_id}")
post_text = html_to_text_with_links(html)

# --- 2) Link history for every section that has appeared in this publication ---
all_section_names = (
    Section.objects.filter(user=user, publication=publication)
    .values_list('section_name', flat=True)
    .distinct()
)

link_history_parts = []
for section_name in sorted(all_section_names):
    # format_link_history expects a Section instance (uses .user, .publication, .section_name)
    # grab any representative Section row for this section_name
    representative = Section.objects.filter(
        user=user, publication=publication, section_name=section_name
    ).first()
    formatted, count = format_link_history(representative)
    if count > 0:
        link_history_parts.append(
            f"=== {section_name} ({count} links) ===\n{formatted}"
        )

link_history_str = "\n\n".join(link_history_parts)

# --- Combine and print ---
output = (
    f"{'='*60}\n"
    f"POST TEXT: {post.title}\n"
    f"{'='*60}\n\n"
    f"{post_text}\n\n"
    f"{'='*60}\n"
    f"LINK HISTORY (all sections)\n"
    f"{'='*60}\n\n"
    f"{link_history_str}"
)

print(output)